# Lab 16 · Learning from feedback: learning curves and trust weights

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. This lab needs no model key.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()

## Warm-up · Learn from feedback

Two reviewers give answers to the same three questions. Compare each with known feedback before assigning more weight. These are synthetic answers, not evidence about real models.

In [ ]:
questions = ['Is an apple a fruit?', 'Is a carrot a fruit?', 'Is a pear a fruit?']
print('Feedback questions:', questions)
expected = ["yes", "no", "yes"]
reviewer_a = ["yes", "no", "yes"]
reviewer_b = ["yes", "yes", "no"]
correct_a, correct_b = 0, 0
for index in range(len(expected)):
    if reviewer_a[index] == expected[index]:
        correct_a += 1
    if reviewer_b[index] == expected[index]:
        correct_b += 1
print("Reviewer A:", correct_a, "/ 3")
print("Reviewer B:", correct_b, "/ 3")

**Next:** Next, turn feedback into weights and evaluate the votes on separate cases.

## Learn which retry reviewer to trust
**Start with a problem:** Equal votes can give too much influence to reviewers that repeatedly make the same mistake.

**In this lab:** Use separate feedback to adjust weights and held-out cases to observe the effect. Small synthetic examples cannot establish real model trust.

**New:** weight a council's votes using agreement with human feedback, and print the resulting learning-curve values for disjoint held-out cases.

The domain question revisits resilience: **is a retry permitted by the operation's idempotency contract?** An idempotent read may be retried within a bounded timeout/backoff policy; a non-idempotent payment must not be blindly repeated. Criticality alone does not make a retry safe. These labels test eligibility, not a full retry or reconciliation policy.

All answers below are **synthetic fixtures**, not recorded model measurements. No model calls, Kafka broker, payment operation, or API key is used.

**At a glance**

![Lab 16 at a glance: Benchmark (retry decisions), then Split once (6 feedback · 4 test), then Feedback case (one per round), then Update trust (per architect), then Weighted vote (test cases only), then Learning curve. Then back to Feedback case: next round.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/16.svg)

## Think about the example

A price lookup is a read, but a card charge changes money and might already have succeeded after a timeout. Feedback cases update trust; separate test cases measure results without influencing the update.

**Concepts:** Retry eligibility and idempotency; Correlated reviewer errors; Feedback/test separation.

Use the scenario above to consider these questions before running the code:

1. Why is retrying a card charge different from retrying a price read?
2. Why must test labels stay out of trust updates?

## The idea in context

Feedback-weighted aggregation changes whom a council trusts. A fixed split and a visible curve make leakage and learning assumptions easier to inspect.

Trust estimates on a tiny dataset are fragile; a rising demo curve is not evidence about real models.

| Key term | Meaning |
| --- | --- |
| **Train/test split** | Separate cases for learning and for scoring. |
| **Learning curve** | The score plotted after each round of feedback. |
| **Weighted vote** | A vote in which each voter counts according to its weight. |
| **Trust** | Here, how often an architect has agreed with the human on the feedback cases. |

## Pattern · Voting-Based Cooperation

Collect independent answers and aggregate them. Weighted votes can reduce a repeated error only if feedback identifies useful reviewers. Correlated mistakes, tiny feedback sets, and ties remain important liabilities. Trust here is empirical agreement on this binary exercise, not a general score of architectural expertise.

## Main exercise at a glance

1. Split the benchmark once
2. Three synthetic reviewers
3. Calculate weights from one feedback case
4. Measure after each feedback round

## Step 1 · Split the benchmark once

Each operation supplies its idempotency contract and whether it is critical. The human labels follow the example's retry restriction. Six cases provide feedback; four remain held out. Never use their labels to update trust or choose weights.

Each tuple contains a case name, whether the operation is idempotent, and whether it is critical. These are synthetic examples, not measured model answers.

In [ ]:
scenarios = [  # (name, idempotent, critical): classroom contracts
    ("Pricing read", True, True), ("Recommendations read", True, False), ("Inventory read", True, True),
    ("Card charge", False, True), ("Refund request", False, True), ("Catalog read", True, False),
    ("Second charge", False, True), ("Stock lookup", True, True), ("Profile lookup", True, False),
    ("Payout request", False, True),
]
expert, feedback, test = {}, [], []
for index, (name, idempotent, critical) in enumerate(scenarios):
    if idempotent:
        expert[name] = "retry permitted"
    else:
        expert[name] = "no blind retry"
    if index < 6:
        feedback.append(name)
    else:
        test.append(name)
assert not set(feedback) & set(test)
print("feedback:", feedback, "\ntest:", test)

## Step 2 · Three synthetic reviewers

The Optimist always allows retries. The Criticality reviewer allows them only for critical work, confusing importance with safety. The Contract reviewer reads idempotency, but deliberately slips on one feedback case. These fixtures make correlated mistakes and calibration visible without claiming any model behaves this way.

In [ ]:
answers = {"Optimist": {}, "Criticality": {}, "Contract": {}}
for name, idempotent, critical in scenarios:
    answers["Optimist"][name] = "retry permitted"
    if critical:
        answers["Criticality"][name] = "retry permitted"
    else:
        answers["Criticality"][name] = "no blind retry"
    if idempotent:
        answers["Contract"][name] = "retry permitted"
    else:
        answers["Contract"][name] = "no blind retry"
answers["Contract"]["Recommendations read"] = "no blind retry"  # deliberate synthetic slip
for name, given in answers.items():
    correct = 0
    for case in test:
        if given[case] == expert[case]:
            correct += 1
    print(name, "correct on", correct, "of", len(test), "held-out cases")

## Step 3 · Calculate weights from one feedback case
Count each reviewer's correct answers on the feedback seen so far. Smoothing starts trust at 0.5 with no evidence. This example turns that agreement into a nonnegative weight. The formula is a teaching choice, not a universal trust score.

In [ ]:
seen = feedback[:1]
weights = {}
for reviewer, given in answers.items():
    correct = 0
    for case in seen:
        if given[case] == expert[case]:
            correct += 1
    trust = (correct + 1) / (len(seen) + 2)
    weights[reviewer] = max(0.0, 2 * trust - 1)
    print(reviewer, "feedback correct:", correct, "weight:", round(weights[reviewer], 2))

## Step 4 · Measure after each feedback round
For each round, calculate the weights from feedback only. Then use those weights to vote on the held-out cases. A tied vote is escalated. Print every result, even if it stays flat or gets worse.

In [ ]:
curve = []
for k in range(len(feedback) + 1):
    weights = {}
    for reviewer, given in answers.items():
        correct = 0
        for case in feedback[:k]:
            if given[case] == expert[case]:
                correct += 1
        trust = (correct + 1) / (k + 2)
        if k == 0:
            weights[reviewer] = 1.0
        else:
            weights[reviewer] = max(0.0, 2 * trust - 1)
    decisions, correct_count = [], 0
    for case in test:
        votes = {"retry permitted": 0.0, "no blind retry": 0.0}
        for reviewer, given in answers.items():
            votes[given[case]] += weights[reviewer]
        if round(votes["retry permitted"], 9) == round(votes["no blind retry"], 9):
            decision = "escalate"
        elif votes["retry permitted"] > votes["no blind retry"]:
            decision = "retry permitted"
        else:
            decision = "no blind retry"
        decisions.append(decision)
        if decision == expert[case]:
            correct_count += 1
    curve.append(correct_count)
    print("round", k, ":", correct_count, "/", len(test), "weights:", weights, "escalated:", decisions.count("escalate"))
print("Learning curve:", curve, "| score change:", curve[-1] - curve[0])

## What you should see

Contract gets all four held-out labels right; Optimist gets two, and Criticality one. At first the two correlated reviewers outvote Contract on non-idempotent charges. After enough feedback, Contract receives the largest weight. The synthetic curve is `[2, 2, 2, 2, 2, 4, 4]`; it illustrates the measurement, not evidence that councils improve in general.

## Try it

1. Move Card charge to the front of the feedback list. Why can a curve based on very few observations rise and then fall?
2. Remove Contract's deliberate slip. When does its weight pull ahead?
3. Give every reviewer the same unsafe answers. Can reweighting discover a correct answer no one proposed?
4. Explain why deduplicating a Kafka message is different from blindly repeating a non-idempotent external charge.
5. **Use it in your council.** Replace fixtures with logged pattern recommendations, keep feedback and test cases separate, and report ties and unsafe recommendations as well as accuracy.

## Takeaway

Feedback can change aggregation, but only held-out results show whether it helped. Idempotency is a business-effects contract: critical operations do not become safe to repeat merely because they are important. A learning curve should reveal failures as honestly as improvements.